# Corrected PEFT final statistical analysis

This CPU-only notebook analyses the frozen test predictions after completion of the validation-selected corrected PEFT Condition C. It does not load a language model, generate predictions, train, select another configuration or overwrite the original five-method analysis.

The original five-method table remains the historical primary comparison. Corrected PEFT is added as an explicitly post-hoc sensitivity condition. Because clauses are clustered within 57 document-title groups, cluster bootstrap intervals are the main robustness analysis. Row-level McNemar results are retained only as descriptive paired checks and are labelled accordingly.


## 1. Load and audit the six frozen prediction files


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

%pip install -q scipy scikit-learn matplotlib


In [ ]:
import json
import os
import re
from datetime import datetime, timezone
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)

DATA_PATH = "/content/drive/MyDrive/Thesis1_Project"
OUTPUT_PREFIX = "corrected_peft_sensitivity"
RANDOM_SEED = 42
NUMBER_OF_BOOTSTRAPS = 5000

CHOICES = [
    "Parties", "License Grant", "Cap On Liability", "Audit Rights",
    "Anti-Assignment", "Insurance", "Document Name", "Agreement Date",
    "Governing Law", "Expiration Date", "Effective Date",
    "Post-Termination Services", "Revenue/Profit Sharing",
    "Minimum Commitment", "Exclusivity", "Rofr/Rofo/Rofn",
    "Ip Ownership Assignment", "Non-Transferable License", "Non-Compete",
    "Termination For Convenience",
]

METHOD_FILES = {
    "TF-IDF + Logistic Regression": {
        "filename": "results_baseline_final.csv",
        "prediction": "predicted_label_baseline",
        "raw": None,
        "status": "primary",
    },
    "Zero-shot": {
        "filename": "results_prompt_engineering_final.csv",
        "prediction": "predicted_label",
        "raw": "raw_output",
        "status": "primary",
    },
    "Original PEFT": {
        "filename": "results_peft_final.csv",
        "prediction": "predicted_label_peft",
        "raw": "raw_output_peft",
        "status": "primary",
    },
    "RAG": {
        "filename": "results_rag_final.csv",
        "prediction": "predicted_label_rag",
        "raw": "raw_output_rag",
        "status": "primary",
    },
    "Hybrid": {
        "filename": "results_rag_peft_hybrid.csv",
        "prediction": "predicted_label_hybrid",
        "raw": "raw_output_hybrid",
        "status": "primary",
    },
    "Corrected PEFT (post-hoc)": {
        "filename": "peft_v4_condition_c_test_final.csv",
        "prediction": "predicted_label",
        "raw": "raw_output",
        "status": "post_hoc_sensitivity",
    },
}
METHOD_ORDER = list(METHOD_FILES)

OUTPUTS = {
    "aligned": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_six_method_predictions.csv"),
    "audit": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_prediction_audit.csv"),
    "comparison": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_comparison.csv"),
    "per_class": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_per_class.csv"),
    "mcnemar": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_row_mcnemar_descriptive.csv"),
    "cluster_ci": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_cluster_bootstrap_cis.csv"),
    "cluster_pairs": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_cluster_pairwise_differences.csv"),
    "cluster_samples": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_cluster_bootstrap_samples.npz"),
    "conflicts": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_conflict_analysis.csv"),
    "parser": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_parser_sensitivity.csv"),
    "summary_json": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_summary.json"),
    "summary_text": os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_summary.txt"),
}
FIGURE_DIR = os.path.join(DATA_PATH, f"{OUTPUT_PREFIX}_figures")
os.makedirs(FIGURE_DIR, exist_ok=True)
OUTPUTS["macro_figure_png"] = os.path.join(FIGURE_DIR, "six_method_macro_f1_cluster_ci.png")
OUTPUTS["macro_figure_svg"] = os.path.join(FIGURE_DIR, "six_method_macro_f1_cluster_ci.svg")
OUTPUTS["class_figure_png"] = os.path.join(FIGURE_DIR, "six_method_per_class_f1.png")
OUTPUTS["class_figure_svg"] = os.path.join(FIGURE_DIR, "six_method_per_class_f1.svg")

test_path = os.path.join(DATA_PATH, "test.csv")
required_paths = [test_path] + [
    os.path.join(DATA_PATH, specification["filename"])
    for specification in METHOD_FILES.values()
]
missing_paths = [path for path in required_paths if not os.path.exists(path)]
if missing_paths:
    raise FileNotFoundError("Required frozen artifacts are missing:\n" + "\n".join(missing_paths))

test_reference = pd.read_csv(test_path).reset_index(drop=True)
required_reference_columns = {"clause_text", "clause_type", "contract_title"}
missing_reference = required_reference_columns - set(test_reference.columns)
if missing_reference:
    raise ValueError(f"test.csv is missing columns: {sorted(missing_reference)}")
if len(test_reference) != 1850:
    raise ValueError(f"Expected 1,850 test rows, found {len(test_reference):,}.")
if test_reference["contract_title"].nunique() != 57:
    raise ValueError(
        f"Expected 57 test document-title groups, found {test_reference['contract_title'].nunique()}."
    )
if set(test_reference["clause_type"].unique()) != set(CHOICES):
    raise ValueError("The test labels do not match the frozen 20-class schema.")
test_reference.insert(0, "row_id", range(len(test_reference)))

aligned = test_reference[["row_id", "clause_text", "clause_type", "contract_title"]].copy()
raw_outputs = {}
audit_records = []

for method_name, specification in METHOD_FILES.items():
    result_path = os.path.join(DATA_PATH, specification["filename"])
    result = pd.read_csv(result_path)
    prediction_column = specification["prediction"]
    required = {"row_id", "clause_text", "clause_type", prediction_column}
    if specification["raw"] is not None:
        required.add(specification["raw"])
    missing = required - set(result.columns)
    if missing:
        raise ValueError(f"{method_name} is missing columns: {sorted(missing)}")

    result["row_id"] = pd.to_numeric(result["row_id"], errors="raise").astype(int)
    if result["row_id"].duplicated().any():
        raise ValueError(f"{method_name} contains duplicate row IDs.")
    result = result.sort_values("row_id").reset_index(drop=True)
    if len(result) != len(test_reference):
        raise ValueError(f"{method_name} has {len(result):,} rows instead of 1,850.")
    if not result["row_id"].equals(test_reference["row_id"]):
        raise ValueError(f"{method_name} row-ID alignment failed.")
    if not result["clause_type"].astype(str).equals(test_reference["clause_type"].astype(str)):
        raise ValueError(f"{method_name} true-label alignment failed.")
    if not result["clause_text"].fillna("").astype(str).equals(
        test_reference["clause_text"].fillna("").astype(str)
    ):
        raise ValueError(f"{method_name} clause-text alignment failed.")

    predictions = result[prediction_column].fillna("Unknown").astype(str)
    invalid = sorted(set(predictions) - set(CHOICES) - {"Unknown"})
    if invalid:
        raise ValueError(f"{method_name} has invalid predictions: {invalid}")

    aligned[method_name] = predictions.to_numpy()
    if specification["raw"] is not None:
        raw_outputs[method_name] = result[specification["raw"]].fillna("").astype(str).to_numpy()
    audit_records.append({
        "method": method_name,
        "analysis_status": specification["status"],
        "filename": specification["filename"],
        "rows": int(len(result)),
        "unknown_count": int(predictions.eq("Unknown").sum()),
        "unknown_rate": float(predictions.eq("Unknown").mean()),
        "alignment": "Passed",
    })

corrected_accuracy = accuracy_score(aligned["clause_type"], aligned["Corrected PEFT (post-hoc)"])
corrected_macro_f1 = f1_score(
    aligned["clause_type"],
    aligned["Corrected PEFT (post-hoc)"],
    labels=CHOICES,
    average="macro",
    zero_division=0,
)
if not np.isclose(corrected_accuracy, 0.8945945945945946):
    raise RuntimeError("Corrected PEFT accuracy does not match the frozen test record.")
if not np.isclose(corrected_macro_f1, 0.8459891223818202):
    raise RuntimeError("Corrected PEFT macro F1 does not match the frozen test record.")

audit_table = pd.DataFrame(audit_records)
aligned.to_csv(OUTPUTS["aligned"], index=False)
audit_table.to_csv(OUTPUTS["audit"], index=False)

print(f"Frozen test rows: {len(aligned):,}")
print(f"Document-title groups: {aligned['contract_title'].nunique()}")
print(f"Methods: {len(METHOD_ORDER)} (five primary + one post-hoc sensitivity)")
display(audit_table)
print("All prediction files passed row, text and label alignment checks.")


## 2. Six-method descriptive results and per-class metrics


In [ ]:
y_true = aligned["clause_type"].astype(str)

comparison_records = []
per_class_records = []
for method_name in METHOD_ORDER:
    y_pred = aligned[method_name].astype(str)
    comparison_records.append({
        "method": method_name,
        "analysis_status": METHOD_FILES[method_name]["status"],
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(
            precision_score(y_true, y_pred, labels=CHOICES, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(y_true, y_pred, labels=CHOICES, average="macro", zero_division=0)
        ),
        "macro_f1": float(
            f1_score(y_true, y_pred, labels=CHOICES, average="macro", zero_division=0)
        ),
        "weighted_f1": float(
            f1_score(y_true, y_pred, labels=CHOICES, average="weighted", zero_division=0)
        ),
        "unknown_count": int(y_pred.eq("Unknown").sum()),
        "unknown_rate": float(y_pred.eq("Unknown").mean()),
    })

    precision, recall, class_f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=CHOICES, zero_division=0
    )
    for label, p, r, score, count in zip(CHOICES, precision, recall, class_f1, support):
        per_class_records.append({
            "method": method_name,
            "analysis_status": METHOD_FILES[method_name]["status"],
            "clause_type": label,
            "support": int(count),
            "precision": float(p),
            "recall": float(r),
            "f1": float(score),
        })

comparison_table = pd.DataFrame(comparison_records)
per_class_table = pd.DataFrame(per_class_records)
comparison_table.to_csv(OUTPUTS["comparison"], index=False)
per_class_table.to_csv(OUTPUTS["per_class"], index=False)

display(comparison_table.round(4))
print("\nRanked by macro F1 (status retained to prevent silent replacement of the primary PEFT run):")
display(comparison_table.sort_values("macro_f1", ascending=False).round(4))

focus_classes = ["Agreement Date", "Effective Date", "License Grant", "Non-Transferable License"]
display(
    per_class_table[per_class_table["clause_type"].isin(focus_classes)]
    .pivot(index="clause_type", columns="method", values="f1")
    .round(4)
)


## 3. Row-level exact McNemar tests — descriptive only

These paired tests reproduce the conventional row-level analysis, but clauses within a document-title group are not independent. Therefore, the p-values below are not the primary inferential evidence. The cluster bootstrap in the next section is the robustness analysis used for thesis interpretation.


In [ ]:
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values), dtype=float)
    running_maximum = 0.0
    for rank, original_index in enumerate(order):
        corrected = min(1.0, (len(p_values) - rank) * p_values[original_index])
        running_maximum = max(running_maximum, corrected)
        adjusted[original_index] = running_maximum
    return adjusted


correctness = {
    method_name: aligned[method_name].astype(str).eq(y_true).to_numpy()
    for method_name in METHOD_ORDER
}
mcnemar_records = []
for method_a, method_b in combinations(METHOD_ORDER, 2):
    a_correct = correctness[method_a]
    b_correct = correctness[method_b]
    a_only = int(np.sum(a_correct & ~b_correct))
    b_only = int(np.sum(~a_correct & b_correct))
    discordant = a_only + b_only
    raw_p = 1.0 if discordant == 0 else float(
        binomtest(a_only, discordant, p=0.5, alternative="two-sided").pvalue
    )
    mcnemar_records.append({
        "method_a": method_a,
        "method_b": method_b,
        "accuracy_a": float(a_correct.mean()),
        "accuracy_b": float(b_correct.mean()),
        "accuracy_difference_b_minus_a": float(b_correct.mean() - a_correct.mean()),
        "a_only_correct": a_only,
        "b_only_correct": b_only,
        "discordant_pairs": discordant,
        "raw_p_value": raw_p,
        "inference_scope": "row_level_descriptive_not_cluster_adjusted",
    })

mcnemar_table = pd.DataFrame(mcnemar_records)
mcnemar_table["holm_adjusted_p_value"] = holm_adjust(mcnemar_table["raw_p_value"])
mcnemar_table["significant_after_holm_0.05"] = (
    mcnemar_table["holm_adjusted_p_value"] < 0.05
)
mcnemar_table.to_csv(OUTPUTS["mcnemar"], index=False)

display(
    mcnemar_table[
        mcnemar_table["method_a"].eq("Corrected PEFT (post-hoc)")
        | mcnemar_table["method_b"].eq("Corrected PEFT (post-hoc)")
    ].round(6)
)
print("Interpret these p-values descriptively; use the cluster-bootstrap intervals below for robustness.")


## 4. Paired cluster bootstrap over 57 document-title groups


In [ ]:
LABEL_TO_ID = {label: index for index, label in enumerate(CHOICES)}
UNKNOWN_ID = len(CHOICES)
NUMBER_OF_OUTPUT_LABELS = len(CHOICES) + 1

true_ids = y_true.map(LABEL_TO_ID).to_numpy(dtype=np.int64)
prediction_ids = {
    method_name: aligned[method_name].map(
        lambda value: LABEL_TO_ID[value] if value in LABEL_TO_ID else UNKNOWN_ID
    ).to_numpy(dtype=np.int64)
    for method_name in METHOD_ORDER
}

group_names = aligned["contract_title"].astype(str).unique()
group_indices = {
    group_name: np.flatnonzero(aligned["contract_title"].astype(str).to_numpy() == group_name)
    for group_name in group_names
}
if len(group_names) != 57:
    raise RuntimeError("Cluster bootstrap requires exactly 57 frozen document-title groups.")

def encoded_metrics(sampled_true, sampled_prediction):
    counts = np.bincount(
        sampled_true * NUMBER_OF_OUTPUT_LABELS + sampled_prediction,
        minlength=len(CHOICES) * NUMBER_OF_OUTPUT_LABELS,
    ).reshape(len(CHOICES), NUMBER_OF_OUTPUT_LABELS)
    true_positive = np.diag(counts[:, :len(CHOICES)]).astype(float)
    true_support = counts.sum(axis=1).astype(float)
    predicted_support = counts[:, :len(CHOICES)].sum(axis=0).astype(float)
    denominator = true_support + predicted_support
    class_f1 = np.divide(
        2.0 * true_positive,
        denominator,
        out=np.zeros_like(true_positive),
        where=denominator != 0,
    )
    return float((sampled_true == sampled_prediction).mean()), float(class_f1.mean())


rng = np.random.default_rng(RANDOM_SEED)
cluster_samples = {
    method_name: {
        "accuracy": np.empty(NUMBER_OF_BOOTSTRAPS),
        "macro_f1": np.empty(NUMBER_OF_BOOTSTRAPS),
    }
    for method_name in METHOD_ORDER
}

for bootstrap_index in range(NUMBER_OF_BOOTSTRAPS):
    sampled_groups = rng.choice(group_names, size=len(group_names), replace=True)
    sampled_indices = np.concatenate([group_indices[group] for group in sampled_groups])
    sampled_true = true_ids[sampled_indices]
    for method_name in METHOD_ORDER:
        accuracy, macro_f1 = encoded_metrics(
            sampled_true, prediction_ids[method_name][sampled_indices]
        )
        cluster_samples[method_name]["accuracy"][bootstrap_index] = accuracy
        cluster_samples[method_name]["macro_f1"][bootstrap_index] = macro_f1
    if (bootstrap_index + 1) % 1000 == 0:
        print(f"Cluster bootstrap: {bootstrap_index + 1:,}/{NUMBER_OF_BOOTSTRAPS:,}")

samples_to_save = {}
for method_name in METHOD_ORDER:
    safe_name = re.sub(r"[^a-z0-9]+", "_", method_name.casefold()).strip("_")
    samples_to_save[f"{safe_name}_accuracy"] = cluster_samples[method_name]["accuracy"]
    samples_to_save[f"{safe_name}_macro_f1"] = cluster_samples[method_name]["macro_f1"]
np.savez_compressed(OUTPUTS["cluster_samples"], **samples_to_save)

cluster_ci_records = []
for method_name in METHOD_ORDER:
    observed = comparison_table.loc[comparison_table["method"].eq(method_name)].iloc[0]
    cluster_ci_records.append({
        "method": method_name,
        "analysis_status": METHOD_FILES[method_name]["status"],
        "clusters": int(len(group_names)),
        "bootstrap_resamples": NUMBER_OF_BOOTSTRAPS,
        "accuracy": float(observed["accuracy"]),
        "accuracy_ci_lower": float(np.percentile(cluster_samples[method_name]["accuracy"], 2.5)),
        "accuracy_ci_upper": float(np.percentile(cluster_samples[method_name]["accuracy"], 97.5)),
        "macro_f1": float(observed["macro_f1"]),
        "macro_f1_ci_lower": float(np.percentile(cluster_samples[method_name]["macro_f1"], 2.5)),
        "macro_f1_ci_upper": float(np.percentile(cluster_samples[method_name]["macro_f1"], 97.5)),
    })
cluster_ci_table = pd.DataFrame(cluster_ci_records)
cluster_ci_table.to_csv(OUTPUTS["cluster_ci"], index=False)

cluster_pair_records = []
for method_a, method_b in combinations(METHOD_ORDER, 2):
    observed_a = comparison_table.loc[comparison_table["method"].eq(method_a)].iloc[0]
    observed_b = comparison_table.loc[comparison_table["method"].eq(method_b)].iloc[0]
    accuracy_difference = (
        cluster_samples[method_b]["accuracy"] - cluster_samples[method_a]["accuracy"]
    )
    macro_difference = (
        cluster_samples[method_b]["macro_f1"] - cluster_samples[method_a]["macro_f1"]
    )
    accuracy_lower, accuracy_upper = np.percentile(accuracy_difference, [2.5, 97.5])
    macro_lower, macro_upper = np.percentile(macro_difference, [2.5, 97.5])
    cluster_pair_records.append({
        "method_a": method_a,
        "method_b": method_b,
        "clusters": int(len(group_names)),
        "bootstrap_resamples": NUMBER_OF_BOOTSTRAPS,
        "accuracy_difference_b_minus_a": float(observed_b["accuracy"] - observed_a["accuracy"]),
        "accuracy_difference_ci_lower": float(accuracy_lower),
        "accuracy_difference_ci_upper": float(accuracy_upper),
        "accuracy_ci_excludes_zero": bool(accuracy_lower > 0 or accuracy_upper < 0),
        "macro_f1_difference_b_minus_a": float(observed_b["macro_f1"] - observed_a["macro_f1"]),
        "macro_f1_difference_ci_lower": float(macro_lower),
        "macro_f1_difference_ci_upper": float(macro_upper),
        "macro_f1_ci_excludes_zero": bool(macro_lower > 0 or macro_upper < 0),
    })
cluster_pair_table = pd.DataFrame(cluster_pair_records)
cluster_pair_table.to_csv(OUTPUTS["cluster_pairs"], index=False)

display(cluster_ci_table.round(4))
print("\nCorrected PEFT paired cluster-bootstrap comparisons:")
display(
    cluster_pair_table[
        cluster_pair_table["method_a"].eq("Corrected PEFT (post-hoc)")
        | cluster_pair_table["method_b"].eq("Corrected PEFT (post-hoc)")
    ].round(4)
)


## 5. Exact-text conflict analysis for all methods


In [ ]:
labels_per_text = aligned.groupby("clause_text")["clause_type"].nunique()
conflicting_texts = set(labels_per_text[labels_per_text > 1].index)
conflict_flag = aligned["clause_text"].isin(conflicting_texts).to_numpy()

conflict_records = []
for method_name in METHOD_ORDER:
    for subset_name, mask in [
        ("all", np.ones(len(aligned), dtype=bool)),
        ("exact_text_conflict", conflict_flag),
        ("no_exact_text_conflict", ~conflict_flag),
    ]:
        subset_true = aligned.loc[mask, "clause_type"].astype(str)
        subset_pred = aligned.loc[mask, method_name].astype(str)
        present_labels = sorted(subset_true.unique())
        conflict_records.append({
            "method": method_name,
            "analysis_status": METHOD_FILES[method_name]["status"],
            "subset": subset_name,
            "rows": int(mask.sum()),
            "labels_present": int(len(present_labels)),
            "accuracy": float(accuracy_score(subset_true, subset_pred)),
            "macro_f1_present_labels": float(f1_score(
                subset_true,
                subset_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )),
            "weighted_f1": float(f1_score(
                subset_true,
                subset_pred,
                labels=present_labels,
                average="weighted",
                zero_division=0,
            )),
        })

conflict_table = pd.DataFrame(conflict_records)
conflict_table.to_csv(OUTPUTS["conflicts"], index=False)

exact_text_ceiling = (
    aligned.groupby(["clause_text", "clause_type"])
    .size()
    .groupby(level=0)
    .max()
    .sum()
    / len(aligned)
)

print(f"Conflicting unique test texts: {len(conflicting_texts)}")
print(f"Rows carrying conflicting exact texts: {int(conflict_flag.sum())}/{len(aligned)}")
print(f"Exact-text deterministic accuracy ceiling: {exact_text_ceiling:.4f}")
display(conflict_table.round(4))


## 6. Strict parser versus leading-valid-label sensitivity

The primary results remain based on the strict parser. This secondary analysis accepts a valid class label only when it appears at the beginning of the generated response, optionally after `Category:` and followed by additional text. It does not search arbitrary substrings and does not change any saved primary prediction.


In [ ]:
LABELS_BY_LENGTH = sorted(CHOICES, key=len, reverse=True)

def parse_leading_valid_label(raw_output):
    if raw_output is None or pd.isna(raw_output):
        return "Unknown"
    text = str(raw_output).strip()
    if not text:
        return "Unknown"
    first_line = text.splitlines()[0].strip()
    first_line = re.sub(r"^[\s`*_\"'“”‘’]+", "", first_line).strip()
    category_match = re.match(r"(?i)^category\s*:\s*(.*)$", first_line)
    if category_match:
        first_line = category_match.group(1).strip()
    for label in LABELS_BY_LENGTH:
        match = re.match(re.escape(label), first_line, flags=re.IGNORECASE)
        if not match:
            continue
        remainder = first_line[match.end():]
        if not remainder or re.match(r"^[\s,;:!?()\[\]{}./\\-]", remainder):
            return label
    return "Unknown"


parser_records = []
for method_name in METHOD_ORDER:
    if method_name not in raw_outputs:
        continue
    strict_predictions = aligned[method_name].astype(str)
    leading_predictions = pd.Series(
        [parse_leading_valid_label(value) for value in raw_outputs[method_name]],
        index=aligned.index,
    )
    for parser_name, predictions in [
        ("strict_primary", strict_predictions),
        ("leading_valid_label_sensitivity", leading_predictions),
    ]:
        parser_records.append({
            "method": method_name,
            "analysis_status": METHOD_FILES[method_name]["status"],
            "parser": parser_name,
            "accuracy": float(accuracy_score(y_true, predictions)),
            "macro_f1": float(f1_score(
                y_true, predictions, labels=CHOICES, average="macro", zero_division=0
            )),
            "weighted_f1": float(f1_score(
                y_true, predictions, labels=CHOICES, average="weighted", zero_division=0
            )),
            "unknown_count": int(predictions.eq("Unknown").sum()),
            "unknown_rate": float(predictions.eq("Unknown").mean()),
        })

parser_table = pd.DataFrame(parser_records)
strict_lookup = parser_table[parser_table["parser"].eq("strict_primary")].set_index("method")
leading_lookup = parser_table[
    parser_table["parser"].eq("leading_valid_label_sensitivity")
].set_index("method")
parser_differences = []
for method_name in strict_lookup.index:
    parser_differences.append({
        "method": method_name,
        "analysis_status": METHOD_FILES[method_name]["status"],
        "strict_accuracy": float(strict_lookup.loc[method_name, "accuracy"]),
        "leading_accuracy": float(leading_lookup.loc[method_name, "accuracy"]),
        "accuracy_change": float(
            leading_lookup.loc[method_name, "accuracy"] - strict_lookup.loc[method_name, "accuracy"]
        ),
        "strict_macro_f1": float(strict_lookup.loc[method_name, "macro_f1"]),
        "leading_macro_f1": float(leading_lookup.loc[method_name, "macro_f1"]),
        "macro_f1_change": float(
            leading_lookup.loc[method_name, "macro_f1"] - strict_lookup.loc[method_name, "macro_f1"]
        ),
        "strict_unknown_count": int(strict_lookup.loc[method_name, "unknown_count"]),
        "leading_unknown_count": int(leading_lookup.loc[method_name, "unknown_count"]),
    })

parser_difference_table = pd.DataFrame(parser_differences)
parser_table.to_csv(OUTPUTS["parser"], index=False)
display(parser_difference_table.round(4))
print("Primary metrics remain the strict-parser results regardless of this sensitivity analysis.")


## 7. Publication-ready sensitivity figures


In [ ]:
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

colors = {
    "TF-IDF + Logistic Regression": "#5B6573",
    "Zero-shot": "#8FBBD9",
    "Original PEFT": "#D98F70",
    "RAG": "#4C78A8",
    "Hybrid": "#59A14F",
    "Corrected PEFT (post-hoc)": "#8B5CF6",
}

plot_data = comparison_table.set_index("method").loc[METHOD_ORDER].reset_index()
ci_lookup = cluster_ci_table.set_index("method")
lower_errors = []
upper_errors = []
for row in plot_data.itertuples():
    lower_errors.append(row.macro_f1 - ci_lookup.loc[row.method, "macro_f1_ci_lower"])
    upper_errors.append(ci_lookup.loc[row.method, "macro_f1_ci_upper"] - row.macro_f1)

figure, axis = plt.subplots(figsize=(11, 6.5))
bars = axis.bar(
    plot_data["method"],
    plot_data["macro_f1"],
    color=[colors[name] for name in plot_data["method"]],
    yerr=np.array([lower_errors, upper_errors]),
    capsize=5,
    edgecolor="black",
    linewidth=0.6,
)
axis.set_ylim(0, 0.95)
axis.set_ylabel("Macro F1")
axis.set_title("Contract clause classification: primary methods and corrected PEFT sensitivity")
axis.tick_params(axis="x", rotation=22)
for label in axis.get_xticklabels():
    label.set_horizontalalignment("right")
for bar, value in zip(bars, plot_data["macro_f1"]):
    axis.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.018,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
axis.text(
    0.01,
    -0.30,
    "Error bars: 95% document-title-group cluster-bootstrap intervals (5,000 resamples). "
    "Corrected PEFT is a post-hoc sensitivity condition and does not silently replace the original PEFT result.",
    transform=axis.transAxes,
    fontsize=8.5,
)
figure.tight_layout()
figure.savefig(OUTPUTS["macro_figure_png"], dpi=300, bbox_inches="tight")
figure.savefig(OUTPUTS["macro_figure_svg"], bbox_inches="tight")
plt.show()

f1_matrix = (
    per_class_table.pivot(index="method", columns="clause_type", values="f1")
    .reindex(index=METHOD_ORDER, columns=CHOICES)
)
figure, axis = plt.subplots(figsize=(22, 7))
image = axis.imshow(f1_matrix.to_numpy(), cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)
axis.set_xticks(np.arange(len(CHOICES)))
axis.set_xticklabels(CHOICES, rotation=55, ha="right", fontsize=9)
axis.set_yticks(np.arange(len(METHOD_ORDER)))
axis.set_yticklabels(METHOD_ORDER, fontsize=10)
axis.set_xlabel("Clause class")
axis.set_ylabel("Method")
axis.set_title("Per-class F1: primary methods and corrected PEFT sensitivity")
for row_index in range(len(METHOD_ORDER)):
    for column_index in range(len(CHOICES)):
        value = f1_matrix.iloc[row_index, column_index]
        axis.text(
            column_index,
            row_index,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=7,
            color="white" if value >= 0.65 else "black",
        )
colorbar = figure.colorbar(image, ax=axis, fraction=0.025, pad=0.02)
colorbar.set_label("F1 score")
figure.tight_layout()
figure.savefig(OUTPUTS["class_figure_png"], dpi=300, bbox_inches="tight")
figure.savefig(OUTPUTS["class_figure_svg"], bbox_inches="tight")
plt.show()

print("Saved macro-F1 figure:", OUTPUTS["macro_figure_png"])
print("Saved per-class figure:", OUTPUTS["class_figure_png"])


## 8. Frozen statistical summary


In [ ]:
corrected_name = "Corrected PEFT (post-hoc)"
baseline_name = "TF-IDF + Logistic Regression"
original_peft_name = "Original PEFT"

def pair_result(method_a, method_b):
    row = cluster_pair_table.loc[
        cluster_pair_table["method_a"].eq(method_a)
        & cluster_pair_table["method_b"].eq(method_b)
    ]
    if len(row) == 1:
        return row.iloc[0].to_dict()
    reverse = cluster_pair_table.loc[
        cluster_pair_table["method_a"].eq(method_b)
        & cluster_pair_table["method_b"].eq(method_a)
    ]
    if len(reverse) != 1:
        raise RuntimeError(f"Missing cluster comparison: {method_a} vs {method_b}")
    source = reverse.iloc[0]
    return {
        **source.to_dict(),
        "method_a": method_a,
        "method_b": method_b,
        "accuracy_difference_b_minus_a": -source["accuracy_difference_b_minus_a"],
        "accuracy_difference_ci_lower": -source["accuracy_difference_ci_upper"],
        "accuracy_difference_ci_upper": -source["accuracy_difference_ci_lower"],
        "macro_f1_difference_b_minus_a": -source["macro_f1_difference_b_minus_a"],
        "macro_f1_difference_ci_lower": -source["macro_f1_difference_ci_upper"],
        "macro_f1_difference_ci_upper": -source["macro_f1_difference_ci_lower"],
    }


corrected_vs_baseline = pair_result(baseline_name, corrected_name)
corrected_vs_original = pair_result(original_peft_name, corrected_name)
corrected_metrics = comparison_table.loc[
    comparison_table["method"].eq(corrected_name)
].iloc[0]

summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "final_corrected_peft_sensitivity_statistics_complete",
    "primary_comparison_preserved": True,
    "corrected_peft_status": "post_hoc_validation_selected_sensitivity",
    "test_rows": int(len(aligned)),
    "document_title_groups": int(len(group_names)),
    "classes": int(len(CHOICES)),
    "bootstrap_resamples": NUMBER_OF_BOOTSTRAPS,
    "corrected_peft_metrics": {
        "accuracy": float(corrected_metrics["accuracy"]),
        "macro_f1": float(corrected_metrics["macro_f1"]),
        "weighted_f1": float(corrected_metrics["weighted_f1"]),
        "unknown_count": int(corrected_metrics["unknown_count"]),
    },
    "corrected_peft_vs_baseline_cluster_bootstrap": corrected_vs_baseline,
    "corrected_peft_vs_original_peft_cluster_bootstrap": corrected_vs_original,
    "exact_text_conflict": {
        "unique_conflicting_texts": int(len(conflicting_texts)),
        "rows": int(conflict_flag.sum()),
        "row_rate": float(conflict_flag.mean()),
        "deterministic_accuracy_ceiling": float(exact_text_ceiling),
    },
    "interpretation_constraints": [
        "Corrected PEFT was developed post hoc after inspection of the original PEFT behaviour.",
        "The original five-method comparison must remain reported.",
        "Cluster-bootstrap intervals are preferred over row-level independence assumptions.",
        "No further model, prompt, parser or hyperparameter selection is permitted after the frozen test evaluation.",
    ],
    "outputs": OUTPUTS,
}

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and np.isnan(value):
        return None
    return value

summary = json_safe(summary)
with open(OUTPUTS["summary_json"], "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False, allow_nan=False)

summary_lines = [
    "CORRECTED PEFT SENSITIVITY - FINAL STATISTICAL SUMMARY",
    "=" * 72,
    "",
    "STATUS",
    "Original five-method comparison: preserved",
    "Corrected PEFT: post-hoc, validation-selected sensitivity analysis",
    "No further model selection permitted",
    "",
    "SIX-METHOD DESCRIPTIVE RESULTS",
    comparison_table.round(4).to_string(index=False),
    "",
    "CLUSTER BOOTSTRAP CONFIDENCE INTERVALS",
    cluster_ci_table.round(4).to_string(index=False),
    "",
    "CORRECTED PEFT VERSUS BASELINE",
    json.dumps(json_safe(corrected_vs_baseline), indent=2),
    "",
    "CORRECTED PEFT VERSUS ORIGINAL PEFT",
    json.dumps(json_safe(corrected_vs_original), indent=2),
    "",
    "EXACT-TEXT CONFLICT DIAGNOSTIC",
    f"Conflicting unique texts: {len(conflicting_texts)}",
    f"Affected rows: {int(conflict_flag.sum())}/{len(aligned)} ({conflict_flag.mean():.2%})",
    f"Deterministic exact-text accuracy ceiling: {exact_text_ceiling:.4f}",
]
with open(OUTPUTS["summary_text"], "w", encoding="utf-8") as file:
    file.write("\n".join(summary_lines))

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nHARD STOP: all test predictions and statistical specifications are frozen.")
print("Do not tune models, prompts, parsers or hyperparameters after reading these outputs.")
print("Send back the comparison table, cluster-CI table, corrected-PEFT pairwise cluster table and parser sensitivity table.")
